# PPO sau SFT/QLoRA cho Qwen2.5-VL-3B-Instruct

Notebook này dùng để huấn luyện bổ sung bằng Reinforcement Learning sau bước SFT/QLoRA.

Pipeline:

```text
Qwen2.5-VL-3B-Instruct
        ↓
Load SFT QLoRA adapter
        ↓
PPO training trên dataset PPO
        ↓
Reward = alpha * VQA Accuracy + beta * BERTScore
        ↓
Lưu PPO LoRA adapter
```

Dữ liệu PPO cần có dạng JSONL:

```json
{"id":"ppo_000001","image":"images/P000002266.jpg","question":"Hoa có màu gì?","reference_answer":"Màu trắng"}
```

Ghi chú: PPO cho VLM khá nặng và dễ OOM trên Colab T4. Lần đầu nên chạy rất nhỏ: `MAX_PPO_SAMPLES = 64`, `PPO_STEPS = 20`.

## 1. Cài thư viện

In [ ]:
!pip -q install -U transformers accelerate peft bitsandbytes qwen-vl-utils pillow tqdm pandas
!pip -q install -U bert-score

In [ ]:
import os
import re
import gc
import json
import math
import random
import string
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from tqdm.auto import tqdm

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Cấu hình đường dẫn

Chỉnh 3 path quan trọng:

- `DATA_ROOT`: thư mục Split chứa `train/images`
- `PPO_JSONL`: file PPO 5000 dòng
- `SFT_ADAPTER_PATH`: adapter QLoRA sau SFT

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path('/content/drive/MyDrive/Final_Deeplearning')
DATA_ROOT = PROJECT_ROOT / 'Split'
PPO_JSONL = PROJECT_ROOT / 'ppo' / 'ppo_train_5000.jsonl'

# Adapter SFT/QLoRA đã train ở notebook trước.
SFT_ADAPTER_PATH = PROJECT_ROOT / 'checkpoints' / 'qwen25_vl_herb_qlora_adapter'

# Nơi lưu adapter sau PPO.
PPO_OUTPUT_DIR = PROJECT_ROOT / 'checkpoints' / 'qwen25_vl_herb_ppo_adapter'
PPO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Fallback nếu chạy local trong repo.
if not DATA_ROOT.exists():
    DATA_ROOT = Path('../').resolve()
if not PPO_JSONL.exists():
    PPO_JSONL = Path('../ppo/ppo_train_5000.jsonl').resolve()

IMAGE_ROOT = DATA_ROOT / 'train'

assert PPO_JSONL.exists(), f'Không thấy PPO_JSONL: {PPO_JSONL}'
assert (IMAGE_ROOT / 'images').exists(), f'Không thấy ảnh train: {IMAGE_ROOT / "images"}'

print('DATA_ROOT:', DATA_ROOT)
print('PPO_JSONL:', PPO_JSONL)
print('IMAGE_ROOT:', IMAGE_ROOT)
print('SFT_ADAPTER_PATH:', SFT_ADAPTER_PATH)
print('PPO_OUTPUT_DIR:', PPO_OUTPUT_DIR)

## 3. Load PPO dataset

In [ ]:
def read_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

ppo_rows = read_jsonl(PPO_JSONL)
print('PPO rows:', len(ppo_rows))
ppo_rows[:2]

In [ ]:
MAX_PPO_SAMPLES = 500  # đổi thành None nếu muốn dùng toàn bộ 5000 dòng
if MAX_PPO_SAMPLES:
    ppo_rows = ppo_rows[:MAX_PPO_SAMPLES]

random.seed(42)
random.shuffle(ppo_rows)
print('Rows used:', len(ppo_rows))

## 4. Load model SFT QLoRA và reference model

PPO cần 2 model:

- `policy_model`: model đang update bằng PPO
- `ref_model`: model tham chiếu cố định để tính KL penalty

Để tiết kiệm VRAM, notebook load cả hai ở 4-bit. Nếu T4 vẫn OOM, giảm batch size và số step.

In [ ]:
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel, LoraConfig, get_peft_model, prepare_model_for_kbit_training
from qwen_vl_utils import process_vision_info

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

policy_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)

if SFT_ADAPTER_PATH.exists():
    policy_model = PeftModel.from_pretrained(policy_model, str(SFT_ADAPTER_PATH), is_trainable=True)
    print('Loaded SFT adapter:', SFT_ADAPTER_PATH)
else:
    print('Không thấy SFT adapter, sẽ gắn LoRA mới trực tiếp lên base model.')
    policy_model = prepare_model_for_kbit_training(policy_model)
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias='none',
        task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    )
    policy_model = get_peft_model(policy_model, lora_config)

policy_model.gradient_checkpointing_enable()
policy_model.train()
policy_model.print_trainable_parameters()

# Reference model cố định: cùng base + cùng SFT adapter, không update.
ref_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
if SFT_ADAPTER_PATH.exists():
    ref_model = PeftModel.from_pretrained(ref_model, str(SFT_ADAPTER_PATH), is_trainable=False)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad_(False)

## 5. Value head cho PPO

PPO cần ước lượng value. Ta thêm một value head nhỏ trên hidden state cuối của token cuối cùng trong câu trả lời.

In [ ]:
hidden_size = policy_model.config.hidden_size
value_head = nn.Linear(hidden_size, 1)
if torch.cuda.is_available():
    value_head = value_head.cuda()

print('hidden_size:', hidden_size)

## 6. Prompt, generate, logprob

In [ ]:
def build_user_text(question):
    return (
        'Bạn là hệ thống hỏi đáp ảnh dược liệu Việt Nam. '
        'Hãy trả lời câu hỏi bằng tiếng Việt, thật ngắn gọn, tối đa 10 từ. '
        'Không giải thích thêm.\n'
        f'Câu hỏi: {question}'
    )

def make_prompt_messages(row):
    image_path = IMAGE_ROOT / row['image']
    return [
        {
            'role': 'user',
            'content': [
                {'type': 'image', 'image': str(image_path)},
                {'type': 'text', 'text': build_user_text(row['question'])},
            ],
        }
    ]

def encode_prompt(row):
    messages = make_prompt_messages(row)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info([messages])
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors='pt',
    )
    if torch.cuda.is_available():
        inputs = inputs.to('cuda')
    return inputs

@torch.inference_mode()
def generate_response(model, row, max_new_tokens=24):
    inputs = encode_prompt(row)
    prompt_len = inputs.input_ids.shape[1]
    generated = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_p=0.9,
        temperature=0.7,
        pad_token_id=processor.tokenizer.eos_token_id,
    )
    response_ids = generated[:, prompt_len:]
    response = processor.batch_decode(response_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()
    return inputs, generated, response, prompt_len

In [ ]:
def sequence_logprob_and_value(model, input_ids, attention_mask, pixel_values=None, image_grid_thw=None, prompt_len=0):
    kwargs = {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'output_hidden_states': True,
        'return_dict': True,
    }
    if pixel_values is not None:
        kwargs['pixel_values'] = pixel_values
    if image_grid_thw is not None:
        kwargs['image_grid_thw'] = image_grid_thw

    out = model(**kwargs)
    logits = out.logits[:, :-1, :]
    labels = input_ids[:, 1:]
    logprobs = F.log_softmax(logits, dim=-1)
    token_logprobs = logprobs.gather(-1, labels.unsqueeze(-1)).squeeze(-1)

    # Các label vị trí >= prompt_len là response tokens.
    response_start = max(prompt_len - 1, 0)
    mask = torch.zeros_like(token_logprobs)
    mask[:, response_start:] = 1.0
    if processor.tokenizer.pad_token_id is not None:
        mask = mask * (labels != processor.tokenizer.pad_token_id).float()

    seq_logprob = (token_logprobs * mask).sum(dim=1)
    last_hidden = out.hidden_states[-1][:, -1, :]
    value = value_head(last_hidden).squeeze(-1)
    return seq_logprob, value

## 7. Reward = VQA Accuracy + BERTScore

Reward được tính:

```text
reward = alpha * vqa_acc + beta * bertscore_f1
```

Trong đó `vqa_acc` là exact match sau chuẩn hóa. Với dataset chỉ có 1 reference answer, soft accuracy VQA v2 suy biến gần như exact match.

In [ ]:
from bert_score import score as bert_score

VI_PUNCT = string.punctuation + '“”‘’…–—。、，！？：；（）[]{}'

def normalize_answer(s):
    if s is None:
        return ''
    s = str(s).lower().strip()
    s = re.sub(r'[{}]'.format(re.escape(VI_PUNCT)), ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def vqa_accuracy(pred, ref):
    p = normalize_answer(pred)
    r = normalize_answer(ref)
    if not p or not r:
        return 0.0
    if p == r:
        return 1.0
    # Nới nhẹ cho đáp án ngắn: "trắng" vs "màu trắng", "có" vs "có, ...".
    if len(r.split()) <= 3 and (r in p or p in r):
        return 1.0
    return 0.0

REWARD_ALPHA = 0.6
REWARD_BETA = 0.4

def compute_rewards(preds, refs):
    accs = [vqa_accuracy(p, r) for p, r in zip(preds, refs)]
    with torch.no_grad():
        _, _, f1 = bert_score(
            preds,
            refs,
            lang='vi',
            model_type='xlm-roberta-base',
            verbose=False,
            rescale_with_baseline=False,
        )
    bert_f1 = f1.cpu().tolist()
    rewards = [REWARD_ALPHA * a + REWARD_BETA * b for a, b in zip(accs, bert_f1)]
    return rewards, accs, bert_f1

## 8. PPO training loop

Đây là PPO tối giản cho VLM:

- Sinh response từ policy hiện tại
- Tính reward bằng VQA Accuracy/BERTScore
- Tính old logprob và ref logprob
- Update LoRA adapter bằng PPO clipped objective + value loss + KL penalty

In [ ]:
PPO_STEPS = 100
PPO_BATCH_SIZE = 1
PPO_EPOCHS = 2
CLIP_EPS = 0.2
VALUE_COEF = 0.5
KL_COEF = 0.02
LR = 5e-6
MAX_NEW_TOKENS = 24

optimizer = torch.optim.AdamW(
    list(filter(lambda p: p.requires_grad, policy_model.parameters())) + list(value_head.parameters()),
    lr=LR,
)

In [ ]:
def get_model_inputs_for_generated(encoded_inputs, generated_ids):
    out = {
        'input_ids': generated_ids,
        'attention_mask': torch.ones_like(generated_ids),
    }
    if 'pixel_values' in encoded_inputs:
        out['pixel_values'] = encoded_inputs['pixel_values']
    if 'image_grid_thw' in encoded_inputs:
        out['image_grid_thw'] = encoded_inputs['image_grid_thw']
    return out

history = []
step_rows = ppo_rows[:]

for step in tqdm(range(PPO_STEPS)):
    row = step_rows[step % len(step_rows)]
    ref_answer = row['reference_answer']

    policy_model.eval()
    encoded, generated_ids, pred, prompt_len = generate_response(policy_model, row, max_new_tokens=MAX_NEW_TOKENS)
    rewards, accs, bert_f1s = compute_rewards([pred], [ref_answer])
    reward = torch.tensor(rewards, dtype=torch.float32, device='cuda' if torch.cuda.is_available() else 'cpu')

    model_inputs = get_model_inputs_for_generated(encoded, generated_ids)

    with torch.no_grad():
        old_logprob, old_value = sequence_logprob_and_value(policy_model, prompt_len=prompt_len, **model_inputs)
        ref_logprob, _ = sequence_logprob_and_value(ref_model, prompt_len=prompt_len, **model_inputs)

    advantage = reward - old_value.detach()

    policy_model.train()
    for _ in range(PPO_EPOCHS):
        new_logprob, value = sequence_logprob_and_value(policy_model, prompt_len=prompt_len, **model_inputs)
        ratio = torch.exp(new_logprob - old_logprob.detach())
        clipped_ratio = torch.clamp(ratio, 1.0 - CLIP_EPS, 1.0 + CLIP_EPS)

        policy_loss = -torch.min(ratio * advantage, clipped_ratio * advantage).mean()
        value_loss = F.mse_loss(value, reward)
        kl = (new_logprob - ref_logprob.detach()).mean()
        loss = policy_loss + VALUE_COEF * value_loss + KL_COEF * kl

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(filter(lambda p: p.requires_grad, policy_model.parameters())) + list(value_head.parameters()), 1.0)
        optimizer.step()

    item = {
        'step': step + 1,
        'id': row['id'],
        'question': row['question'],
        'reference_answer': ref_answer,
        'prediction': pred,
        'reward': float(reward.item()),
        'vqa_acc': float(accs[0]),
        'bertscore_f1': float(bert_f1s[0]),
        'loss': float(loss.detach().cpu().item()),
    }
    history.append(item)

    if (step + 1) % 10 == 0:
        recent = pd.DataFrame(history[-10:])
        print({
            'step': step + 1,
            'reward': recent['reward'].mean(),
            'vqa_acc': recent['vqa_acc'].mean(),
            'bertscore_f1': recent['bertscore_f1'].mean(),
        })

    if (step + 1) % 50 == 0:
        policy_model.save_pretrained(str(PPO_OUTPUT_DIR))
        processor.save_pretrained(str(PPO_OUTPUT_DIR))
        torch.save(value_head.state_dict(), PPO_OUTPUT_DIR / 'value_head.pt')
        pd.DataFrame(history).to_csv(PPO_OUTPUT_DIR / 'ppo_training_log.csv', index=False, encoding='utf-8-sig')

policy_model.save_pretrained(str(PPO_OUTPUT_DIR))
processor.save_pretrained(str(PPO_OUTPUT_DIR))
torch.save(value_head.state_dict(), PPO_OUTPUT_DIR / 'value_head.pt')
pd.DataFrame(history).to_csv(PPO_OUTPUT_DIR / 'ppo_training_log.csv', index=False, encoding='utf-8-sig')
print('Saved PPO adapter:', PPO_OUTPUT_DIR)

## 9. Xem log huấn luyện

In [ ]:
log_df = pd.DataFrame(history)
log_df.tail(10)

In [ ]:
if len(log_df):
    print('Mean reward:', log_df['reward'].mean())
    print('Mean VQA acc:', log_df['vqa_acc'].mean())
    print('Mean BERTScore F1:', log_df['bertscore_f1'].mean())

## 10. Gợi ý báo cáo

Trong báo cáo có thể mô tả:

> Sau khi fine-tune SFT bằng QLoRA, nhóm tiếp tục tối ưu mô hình bằng PPO. Reward được thiết kế kết hợp giữa độ chính xác VQA sau chuẩn hóa và BERTScore-F1 để cân bằng giữa khớp đáp án ngắn và tương đồng ngữ nghĩa. KL penalty với mô hình SFT ban đầu được dùng để hạn chế mô hình lệch khỏi hành vi đã học sau SFT.

Khi đánh giá chính thức, dùng lại notebook evaluation giống B1/B2 nhưng load adapter PPO tại `PPO_OUTPUT_DIR`.